In [9]:
from pathlib import Path
import pandas as pd
from HGM import HGMST

base = Path("../Data/1.DLPFC")
slices = sorted([str(p).replace("\\", "/") + "/" for p in base.iterdir() if p.is_dir()])
test_slices = [slices[i] for i in [0, 7, -4, -3, -2, -1]]
use_test_slices = False
if use_test_slices:
    slices = test_slices

In [10]:
methods = [
    {"name": "base", "use_fused": False},
    # {"name": "fusion_contrast", "use_fused": True, "fuse_mode": "adaptive_weight", "fused_con_weight": 0.1, "fusion_head": "attention"},
    {"name": "dgi_anchor", "use_fused": True, "fuse_mode": "adaptive_weight", "dgi": 0.1, "fusion_head": "attention"},
    # 可选：保留可学习权重作为参考
    {"name": "learnable_weight", "use_fused": True, "fuse_mode": "adaptive_weight", "dgi": 0.1, "fusion_head": "learnable"},
]

# 快速试验开关：True 则用少量 epoch 和切片，便于快速比较
quick = False
epochs = 30 if quick else 100
use_test_slices_for_quick = True
records = []
for method in methods:
    for path in slices:
        if quick and not use_test_slices_for_quick:
            continue
        try:
            cfg = {
                "use_fused": method.get("use_fused", False),
                "fuse_mode": method.get("fuse_mode", "neighbor_union"),
                "fusion_head": method.get("fusion_head", "attention"),
                "dgi": method.get("dgi", 0),
                "fused_con_weight": method.get("fused_con_weight", 0.0),
            }
            print(f"Running {method['name']} on {path.split('/')[-2]} (dgi={cfg['dgi']}, fused_con={cfg['fused_con_weight']})")
            hgm = HGMST(
                path,
                seed=2020,
                radius=18,
                use_fused=cfg["use_fused"],
                fuse_mode=cfg["fuse_mode"],
                fusion_head=cfg["fusion_head"],
                dgi=cfg["dgi"],
                fused_con_weight=cfg["fused_con_weight"],
            )
            hgm.train(epochs=epochs)
            _, res_df = hgm.eval(refine_radius=300)
            m = res_df.loc["mclust_smooth", ["ARI", "NMI", "FMI"]]
            records.append({
                "method": method["name"],
                "slice": path.split("/")[-2],
                "ARI": float(m["ARI"]),
                "NMI": float(m["NMI"]),
                "FMI": float(m["FMI"]),
            })
        except Exception as e:
            print(f"Error running {method['name']} on {path}: {e}")
            records.append({
                "method": method["name"],
                "slice": path.split("/")[-2],
                "ARI": float("nan"),
                "NMI": float("nan"),
                "FMI": float("nan"),
            })

metrics = pd.DataFrame(records).sort_values(["method", "slice"]).reset_index(drop=True)
summary = (
    metrics.groupby("method")[ ["ARI", "NMI", "FMI"] ]
    .agg(["mean", "std", "median", "max", "min"])
    .reset_index()
 )
summary.columns = [
    col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
    for col in summary.columns.to_flat_index()
 ]
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Running base on 151507 (dgi=0, fused_con=0.0)


100%|██████████| 100/100 [00:03<00:00, 27.64it/s]


fitting ...
  |======================================================================| 100%
Running base on 151508 (dgi=0, fused_con=0.0)


100%|██████████| 100/100 [00:03<00:00, 26.13it/s]


fitting ...
  |======================================================================| 100%
Running base on 151509 (dgi=0, fused_con=0.0)


...

fitting ...
  |======================================================================| 100%
Running learnable_weight on 151675 (dgi=0.1, fused_con=0.0)


100%|██████████| 100/100 [00:08<00:00, 12.03it/s]


fitting ...
  |======================================================================| 100%
Running learnable_weight on 151676 (dgi=0.1, fused_con=0.0)


100%|██████████| 100/100 [00:08<00:00, 12.10it/s]


fitting ...
  |======================================================================| 100%
          method  ARI_mean  ARI_std  ARI_median  ARI_max  ARI_min  NMI_mean  NMI_std  NMI_median  NMI_max  NMI_min  FMI_mean  FMI_std  FMI_median  FMI_max  FMI_min
            base    0.5156   0.0909      0.5238   0.7099   0.3868    0.6287   0.0540      0.6540   0.6862   0.5560    0.6264   0.0951      0.6310   0.8430   0.4904
      dgi_anchor    0.4928   0.0883      0.5029   0.5881   0.3269    0.6270   0.0538      0.6377   0.6864   0.5040    0.6079   0.0925      0.6359   0.7405   0.4391
learnable_weight    0.4892   0.0522      0.4710   0.5877   0.4191    0.6171   0.0419      0.6128   0.6815   0.5620    0.6052   0.0615      0.6020   0.6928   0.5191
